# 🔬 Mini-Project 11: 3D IC AI Chiplet Thermal Floorplanning & Dark-Silicon Guard
**PravahaQ (`v0.1.0`) — Semiconductor EDA Showcase (3-Body Cubic *Tryāṇuka* HUBO + OpenQASM 3.0)**

---

## 🟢 Layer 1: Plain-English Intuition

Modern AI accelerators (like AMD MI300X or NVIDIA Blackwell) stack multiple **Chiplets** (Compute dies, HBM3e memory stacks, and I/O dies) onto a silicon interposer.
- **The Trade-Off:** Placing two high-bandwidth compute chiplets right next to each other minimizes wire delay (picojoules per bit), **but** creates a localized **thermal hotspot**.
- **The 3-Body Hotspot ($K_{ijk} \hat{Z}_i \hat{Z}_j \hat{Z}_k$):** When two adjacent compute chiplets *and* the underlying HBM memory stack all operate at boost clock simultaneously, vertical heat flux exceeds the vapor-chamber cooling limit ($110^\circ\text{C}$), triggering **Dark Silicon emergency throttling**.

Here we model 3D chiplet thermal coupling directly via our ***Vaisheshika* 3-Body Cubic HUBO** and export the hardware circuit to **OpenQASM 3.0** and **NVIDIA CUDA-Q**.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd() / "src"))

from pravaha_q.math_engine import HUBOPolynomial, hubo_to_ising, analyze_ising_spectrum
from pravaha_q.algorithms import BFDCQOEngine
from pravaha_q.compiler import export_cudaq_kernel_source

# 8 binary variables representing 8 candidate 3D interposer die slots / boost states
chip_hubo = HUBOPolynomial(num_variables=8)
for i, wire_reward in enumerate([-3.5, -2.8, -4.1, -3.0, -3.6, -2.9, -3.2, -3.8]):
    chip_hubo.add_linear(i, wire_reward)

# Pairwise interconnect affinity vs lateral heat bleed
for (u, v), q_val in [((0, 1), 3.2), ((2, 3), 2.8), ((4, 5), -1.5), ((0, 2), -2.0), ((1, 5), 2.5)]:
    chip_hubo.add_quadratic(u, v, q_val)

# 3-Body Vertical Stack Thermal Hotspot Penalties (Compute_A + Compute_B + HBM_Stack_C)
chip_hubo.add_cubic(0, 2, 4, 9.0)
chip_hubo.add_cubic(1, 3, 5, 8.5)
chip_hubo.add_cubic(2, 4, 6, 7.5)

chip_ising = hubo_to_ising(chip_hubo)
chip_spec = analyze_ising_spectrum(chip_ising)
bf_res = BFDCQOEngine(shots=1024, seed=42).run(chip_ising)
cudaq_src, qir_prof = export_cudaq_kernel_source(chip_ising)

print("=== Mini-Project 11: 3D IC Chiplet Thermal Floorplanning (3-Local HUBO) ===")
print(f"Active 3-Body Thermal Triads : {len(chip_ising.k_cubic)} Cubic Tryanuka Terms")
print(f"Optimal Floorplan Bitstring  : |{chip_spec.ground_state_bitstring}>")
print(f"Ground-State Energy          : {chip_spec.ground_energy:.4f} (BF-DCQO Found: {bf_res.best_energy:.4f})")
print(f"QIR Base Profile Instructions: {qir_prof.quantum_instruction_count} Quantum Instructions | {qir_prof.two_qubit_entangling_count} Two-Qubit Gates\n")
print("--- Generated NVIDIA CUDA-Q Kernel Preview ---")
print("\n".join(cudaq_src.splitlines()[:8]))

=== Mini-Project 11: 3D IC Chiplet Thermal Floorplanning (3-Local HUBO) ===
Active 3-Body Thermal Triads : 3 Cubic Tryanuka Terms
Optimal Floorplan Bitstring  : |10101101>
Ground-State Energy          : -19.2500 (BF-DCQO Found: -19.2500)
QIR Base Profile Instructions: 58 Quantum Instructions | 28 Two-Qubit Gates

--- Generated NVIDIA CUDA-Q Kernel Preview ---
"""Auto-generated by PravahaQ v0.1.0 — NVIDIA CUDA-Q Hybrid Kernel."""
import cudaq

@cudaq.kernel
def pravaha_bf_dcqo_kernel() -> None:
    q = cudaq.qvector(8)
    h(q)
